# MiniMax H3 — 静止画1枚から縦動画

上から順に実行すると、Drive の `マイドライブ/h3-runner/output/orbis01.mp4` に 15秒・1080×1920 の動画が出ます。

- ランタイムは GPU。A100 40GB でも、RTX PRO 6000（表示 95GB 前後）でも進みます。24GB 未満は止まります。
- 重みはローカルの `/content/hf-cache`。Drive には完成 mp4 だけ書きます。
- 参照動画は使いません。サクラの静止画1枚 + 英語プロンプトです。
- 15秒を1本では出しません。6秒（文章だけ）と 9秒（静止画）を作り、ffmpeg で 15秒にします。Ref2VA の画は 352×640 のままです。理由は計画のセルが出します。
- LoRA は要りません。追加のモデルも取りません。
- 秘密情報は入れません。Hugging Face のライセンス同意が要るときだけ、Colab のシークレット `HF_TOKEN` を使います。画面には出しません。

[Open in Colab](https://colab.research.google.com/github/fireworker011/Research/blob/cursor/h3-still-to-video-7cb8/h3-runner/minimax_h3_still.ipynb)


## 1. Drive と GPU

Drive をマウントします。重み用の空きはローカル（`/content`）、完成 mp4 は Drive です。GPU が無い、または 24GB 未満なら止まります。


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/h3-runner/output")
CACHE = Path("/content/hf-cache")
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE / "hub")
os.environ["H3_HF_CACHE"] = str(CACHE)
os.environ["H3_OUT_DIR"] = str(OUT)

token = ""
try:
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN をシークレットから読みました（値は表示しません）")
else:
    print("HF_TOKEN は未設定。公開のまま落ちるときは、ライセンス同意後にシークレットへ入れてこのセルからやり直す。")

for label, disk in (("ローカル", "/content"), ("Drive", "/content/drive")):
    usage = shutil.disk_usage(disk)
    print(f"{label} {disk}: 空き {usage.free / 1e9:.1f} GB / 全体 {usage.total / 1e9:.1f} GB（10^9）")

if shutil.which("nvidia-smi"):
    subprocess.check_call(["nvidia-smi", "-L"])
import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU がオフです。ランタイムのタイプを GPU にして、このセルからやり直してください。")
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
mem = Path("/proc/meminfo").read_text(encoding="utf-8")
host = int(next(line.split()[1] for line in mem.splitlines() if line.startswith("MemTotal:"))) / 1024 ** 2
print(f"GPU: {props.name}  VRAM: {vram:.1f} GB  ホストRAM: {host:.1f} GB")
os.environ["H3_VRAM_GB"] = f"{vram:.4f}"
os.environ["H3_HOST_RAM_GB"] = f"{host:.4f}"
if vram < 24:
    raise SystemExit(
        "VRAM が 24GB 未満です。公式の int8 offload は 24〜32GB 向けです。"
        "A100 か RTX PRO 6000 に変えて、このセルからやり直してください。"
    )
if vram < 70:
    print("40GB クラスです。公式の int8 + group offload で続けます。")
elif host >= 140:
    print("80GB 以上かつホスト RAM 140GB 以上です。bf16 offload で続けます。")
else:
    print("80GB 以上です。ホスト RAM が約 140GB 未満なので int8 offload で続けます。")
print("重みキャッシュ:", CACHE, " 完成 mp4:", OUT)


## 2. パッケージ

公式の diffusers 実装を入れます。Colab に入っている CUDA 版の torch は残します。


In [ ]:
import shutil
import subprocess
import sys

if shutil.which("ffmpeg") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])

# Colab の CUDA 付き torch は置き換えない。diffusers は公式クラスが入っている main のコミット。
pkgs = [
    "git+https://github.com/huggingface/diffusers.git@5ff8e59ff9fe81c6e2df4fb4c6ea0d97a5df5ab2",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "safetensors>=0.4.3",
    "huggingface_hub>=0.25.0",
    "torchao",
    "av>=11.0.0",
    "imageio>=2.34.0",
    "imageio-ffmpeg>=0.5.0",
    "soundfile>=0.12.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs])
print("install ok", shutil.which("ffmpeg"))


## 3. ランナー

プロンプトとサクラの静止画はこのリポジトリに入っています。


In [ ]:
import subprocess
from pathlib import Path

dest = Path("/content/h3-runner")
branch = "cursor/h3-still-to-video-7cb8"
repo = "https://github.com/fireworker011/Research.git"
if not (dest / "run_h3.py").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(dest)])
else:
    subprocess.check_call(["git", "-C", str(dest), "fetch", "--depth", "1", "origin", branch])
    subprocess.check_call(["git", "-C", str(dest), "checkout", branch])
    subprocess.check_call(["git", "-C", str(dest), "pull", "--ff-only", "origin", branch])
print("runner", dest / "run_h3.py")


## 4. 計画

重みはまだ落としません。子プロセスの標準出力と標準エラーをこのセルに出します。終了コードが 0 でなければ stderr 全文を出して止まります。


In [ ]:
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

script = Path("/content/h3-runner/run_h3.py")
if not script.is_file():
    raise SystemExit(f"ランナーが無い: {script}。前のセルからやり直す。")
cmd = [
    sys.executable,
    "-u",
    str(script),
    "--preset", "orbis01",
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--vram-gb", os.environ["H3_VRAM_GB"],
    "--host-ram-gb", os.environ["H3_HOST_RAM_GB"],
    "--seed", "0",
    "--dry-run",
]
run_logged(cmd)
print("計画は上の通り。次のセルで生成する。")


## 5. 生成

ここが本処理です。最初の段は約 149GB（未完了シャードを含む）。足りなければダウンロード前に止まります。T2VA のあと `transformer/` を消してから `transformer_ref/` を取ります。終わると `orbis01.mp4` のパスを表示します。


In [ ]:
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

script = Path("/content/h3-runner/run_h3.py")
if not script.is_file():
    raise SystemExit(f"ランナーが無い: {script}。前のセルからやり直す。")
cmd = [
    sys.executable,
    "-u",
    str(script),
    "--preset", "orbis01",
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--vram-gb", os.environ["H3_VRAM_GB"],
    "--host-ram-gb", os.environ["H3_HOST_RAM_GB"],
    "--seed", "0",
]
run_logged(cmd)
out = Path(os.environ["H3_OUT_DIR"]) / "orbis01.mp4"
if not out.is_file():
    raise SystemExit(f"完成 mp4 がありません: {out}")
print("完成:", out, "bytes", out.stat().st_size)
